# Run the matched-budget GST seed sweep

For each seed this runs adaptive FPR first, uses its accounted revealed-shot cost as the target, and calibrates the two fixed-shot comparisons toward that same budget.

In [ ]:
from pathlib import Path
import subprocess
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / "seed_sweep_experiments",
    Path.cwd() / "GST_POUNDERS" / "seed_sweep_experiments",
    Path("/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments"),
]
EXPERIMENT_DIR = next(
    (p.resolve() for p in candidates if (p / "run_matched_budget_sweep.py").exists()),
    None,
)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Could not locate seed_sweep_experiments.")

CONFIG_PATH = EXPERIMENT_DIR / "experiment_config.json"
RESULTS_DIR = EXPERIMENT_DIR / "matched_results"
print("Experiment directory:", EXPERIMENT_DIR)
print("Python:", sys.executable)

In [ ]:
RUN_SWEEP = True
SEED_SPEC = "101:102"          # Expand after one seed succeeds.
FORCE_RERUN = True
# Choose which methods to run. fixed_fpr is the budget ANCHOR -- it is auto-run/loaded
# whenever adaptive_fpr or fixed_no_fpr is selected (they need its matched budget).
METHODS = ["fixed_fpr", "adaptive_fpr"] #"fixed_fpr", "adaptive_fpr", "fixed_no_fpr"]
# rho_gated_geometric , lazy_delta_inverse_square

In [ ]:
command = [
    sys.executable, "-u", str(EXPERIMENT_DIR / "run_matched_budget_sweep.py"),
    "--config", str(CONFIG_PATH),
    "--results-dir", str(RESULTS_DIR),
    "--seeds", SEED_SPEC,
    "--methods", ",".join(METHODS),
]
if FORCE_RERUN:
    command.append("--force")

print("Command:", " ".join(command))
if not RUN_SWEEP:
    print("Dry run only. Set RUN_SWEEP=True to execute.")
else:
    process = subprocess.Popen(
        command,
        cwd=EXPERIMENT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Matched sweep exited with code {return_code}")
    print("Matched-budget sweep complete.")

In [ ]:
import pandas as pd

summary_path = RESULTS_DIR / "matched_budget_summary.csv"
if summary_path.exists():
    matched_summary_df = pd.read_csv(summary_path)
    display(
        matched_summary_df[ 
            [
                "data_seed",
                "method",
                "target_revealed_shots",
                "actual_revealed_shots",
                "relative_budget_difference",
                "weighted_least_squares_objective",
                "mean_gate_entanglement_infidelity_to_truth",
                "mean_spam_vector_l2_error_to_truth",
            ]
        ].sort_values(["data_seed", "method"]).reset_index(drop=True)
    )
else:
    print("No matched-budget summary exists yet.")

Open analyze_detailed_accuracy.ipynb and set RESULTS_DIR to matched_results to plot the matched-budget runs.